In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

C:\Users\omole\AppData\Local\Temp\ipykernel_16704\4084895585.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
# wikipedia API wrapper
wikipedia_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=wikipedia_api_wrapper)

print(wiki.name)

wikipedia


In [3]:
arxiv_api_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)
print(arxiv.name)

arxiv


In [4]:
tools = [wiki, arxiv]

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

USER_AGENT environment variable not set, consider setting it to identify your requests.


True

In [6]:
import os
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

In [7]:
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

c:\Users\omole\OneDrive\Desktop\langchain_projects\Q_A_project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6063.00it/s]


In [8]:
loader = WebBaseLoader('https://docs.smith.langchain.com')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()

In [9]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(retriever, 'Langsmith-search', 'Search any information about Langsmith')

In [10]:
tools = [arxiv, wiki, retriever_tool]
tools

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\omole\\OneDrive\\Desktop\\langchain_projects\\Q_A_project\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='Langsmith-search', description='Search any information about Langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001CBB74BA560>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001CBB756A7A0>)]

In [11]:
from langchain_groq import ChatGroq

In [12]:
groq_api_key = os.environ['GROQ_API_KEY']

llm = ChatGroq(model='openai/gpt-oss-120b', groq_api_key=groq_api_key)

In [13]:
from langchain_classic.agents import (AgentExecutor, create_openai_functions_agent)
from langsmith import Client 

# 2. Instantiate the LangSmith client
client = Client()

# 3. Pull the prompt directly via the client object
prompt = client.pull_prompt('hwchase17/openai-functions-agent', dangerously_pull_public_prompt=True)

In [14]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [15]:
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm, tools, prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [16]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [17]:
agent_executor.invoke({'input': 'Tell me about Langsmith'})



> Entering new AgentExecutor chain...

Invoking: `Langsmith-search` with `{'query': 'Langsmith'}`


root cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

LangSmith Observability - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance 

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith – A quick‑start guide**\n\nLangSmith is the **observability, evaluation, and workflow‑automation platform** that LangChain built for production‑grade LLM (large‑language‑model) applications. Think of it as the “Datadog” or “New\u202fRelic” for AI: it records everything your LLM‑powered code does, surfaces problems, lets you measure quality, and helps you close the loop with automated fixes.\n\n---\n\n## 1. What LangSmith does\n\n| Category | What you get | Why it matters |\n|----------|--------------|----------------|\n| **Tracing & Observability** | Every request, prompt, response, tool call, and internal “turn” is stored as a **trace**. You can drill‑down into individual runs, view timelines, and compare versions. | Debug failures fast, understand latency bottlenecks, and keep an audit trail for compliance. |\n| **Evaluation** | Built‑in metrics (e.g., relevance, factuality, toxicity) plus a UI for custom evaluation functio

In [18]:
agent_executor.invoke({'input': 'Breast Cancer'})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Breast cancer'}`


Page: Breast cancer
Summary: Breast cancer is a cancer that develops from breast tissue. Signs of breast cancer may include: a lump in the breast, a change in breast shape, dimpling of the skin, milk rejection, fluid coming from the nipple, a newly i**Breast Cancer – Overview**

**What it is**  
Breast cancer is a malignant (cancerous) disease that originates in the cells of the breast. It can arise in both women and, far less commonly, men. The most common type begins in the lining of the milk ducts (ductal carcinoma) or the lobules that produce milk (lobular carcinoma).

---

### 1. Types & Classification
| Category | Sub‑type | Typical Features |
|----------|----------|-------------------|
| **By origin** | *Ductal carcinoma in situ* (DCIS) | Non‑invasive, confined to ducts |
| | *Lobular carcinoma in situ* (LCIS) | Non‑invasive, confined to lobules |
| | *Invasive ductal carcinoma* (I

{'input': 'Breast Cancer',
 'output': '**Breast Cancer – Overview**\n\n**What it is**  \nBreast cancer is a malignant (cancerous) disease that originates in the cells of the breast. It can arise in both women and, far less commonly, men. The most common type begins in the lining of the milk ducts (ductal carcinoma) or the lobules that produce milk (lobular carcinoma).\n\n---\n\n### 1. Types & Classification\n| Category | Sub‑type | Typical Features |\n|----------|----------|-------------------|\n| **By origin** | *Ductal carcinoma in situ* (DCIS) | Non‑invasive, confined to ducts |\n| | *Lobular carcinoma in situ* (LCIS) | Non‑invasive, confined to lobules |\n| | *Invasive ductal carcinoma* (IDC) | ~70‑80\u202f% of invasive cancers; forms a solid tumor |\n| | *Invasive lobular carcinoma* (ILC) | Often diffuse, harder to detect on imaging |\n| **By hormone‑receptor status** | *Hormone‑receptor‑positive* (ER⁺/PR⁺) | Grows in response to estrogen/progesterone; treatable with endocrine the

In [19]:
agent_executor.invoke({'input': 'what is the paper 1706.03762 about'})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`




AttributeError: 'Search' object has no attribute 'results'